# Linear Algebra in Transformers

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 01.01–01.09 |
| Time | ~90 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/10_linear_algebra_in_transformers.ipynb)

---

## 🎯 Learning Objective

Walk through the transformer architecture end-to-end through a purely linear-algebra lens: Q/K/V projections, scaled dot-product attention, multi-head concatenation, and residual connections.

---

## 📐 Theory

This notebook is the synthesis of Module 01: we walk through one transformer block end-to-end,
naming which linear-algebra concept powers each step.

### Step 1 — Embeddings ([01.01](01_vectors_and_spaces.ipynb))

Each token is looked up as a row of an embedding matrix, becoming a vector in
$\mathbb{R}^{d_{\text{model}}}$.

### Step 2 — Q/K/V projections ([01.02](02_matrix_operations.ipynb), [01.03](03_linear_transformations.ipynb))

Three learned weight matrices linearly transform every token embedding $\mathbf{x}$ into a
query, key, and value:

$$\mathbf{q} = W_Q\mathbf{x} \qquad \mathbf{k} = W_K\mathbf{x} \qquad \mathbf{v} = W_V\mathbf{x}$$

### Step 3 — Scaled dot-product attention ([01.01](01_vectors_and_spaces.ipynb), [01.09](09_tensor_operations.ipynb))

Stacking every token's query/key/value as rows of matrices $Q,K,V$ (shape
`(seq_len, d_k)`), attention is:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

$QK^\top$ computes *every pairwise dot product* between queries and keys at once — a grid of
cosine-similarity-like scores (Notebook 01.01), scaled by matrix multiplication
(Notebook 01.02). Dividing by $\sqrt{d_k}$ keeps the dot products from growing too large as
dimensionality increases (we derive exactly why in
[Notebook 03.03](../03_Probability_and_Statistics/03_expectation_variance_moments.ipynb)'s
treatment of variance scaling, $\text{Var}(aX)=a^2\text{Var}(X)$).
The softmax turns scores into a weighted average recipe, and multiplying by $V$ mixes value
vectors according to those weights.

### Step 4 — Multi-head attention ([01.09](09_tensor_operations.ipynb))

Rather than one attention computation in the full $d_{\text{model}}$ dimensions, the model
splits into $h$ heads of dimension $d_k = d_{\text{model}}/h$, runs attention independently per
head (a batched tensor contraction), then concatenates and linearly re-projects the results.

### Step 5 — Residual connections ([01.03](03_linear_transformations.ipynb))

Each sub-layer's output is *added back* to its input: $\mathbf{x}_{\text{out}} = \mathbf{x} +
\text{Sublayer}(\mathbf{x})$. In transformation terms, this is "identity transformation plus a
learned correction" — it guarantees a direct gradient path all the way back to the input, no
matter how deep the network, because the identity part of $\mathbf{x}_{\text{out}}$ always
carries gradient straight through.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

An attention weight matrix is literally a heatmap of "how much does query position $i$ attend
to key position $j$" — visualizing it makes the abstract $QK^\top$ formula into a picture of
which tokens are talking to which.

In [ ]:
# Visualizing an attention weight matrix as a token-to-token heatmap
import numpy as np
import matplotlib.pyplot as plt

def softmax_rows(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

tokens = ["The", "cat", "sat", "on", "the", "mat"]
seq_len, d_k = len(tokens), 8
rng = np.random.default_rng(3)

Q = rng.standard_normal((seq_len, d_k))
K = rng.standard_normal((seq_len, d_k))
scores = (Q @ K.T) / np.sqrt(d_k)
weights = softmax_rows(scores)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap="viridis")
ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens)
ax.set_xlabel("Key position (attended TO)"); ax.set_ylabel("Query position (attending FROM)")
ax.set_title(r"softmax($QK^T / \sqrt{d_k}$): each row sums to 1")
plt.colorbar(im, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()
print("Each row sums to 1 (it's a probability distribution over 'where to look'):", np.allclose(weights.sum(axis=1), 1.0))

## 🐍 Implementation from Scratch

A complete single-head scaled dot-product attention function, then multi-head attention built
by batching that single-head computation across a head axis using the einsum patterns from
[Notebook 01.09](09_tensor_operations.ipynb).

In [ ]:
# Full scaled dot-product attention + multi-head attention from scratch
import numpy as np

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    # Q, K, V: shape (..., seq_len, d_k). Returns (..., seq_len, d_v) and the attention weights.
    d_k = Q.shape[-1]
    scores = np.einsum('...id,...jd->...ij', Q, K) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    output = np.einsum('...ij,...jd->...id', weights, V)
    return output, weights

def multi_head_attention(x, Wq, Wk, Wv, Wo, n_heads):
    # x: (seq_len, d_model). Wq/Wk/Wv/Wo: (d_model, d_model). Splits into n_heads internally.
    seq_len, d_model = x.shape
    d_k = d_model // n_heads

    Q, K, V = x @ Wq, x @ Wk, x @ Wv                     # (seq_len, d_model) each

    def split_heads(M):
        return M.reshape(seq_len, n_heads, d_k).transpose(1, 0, 2)   # -> (n_heads, seq_len, d_k)

    Qh, Kh, Vh = split_heads(Q), split_heads(K), split_heads(V)
    out_heads, _ = scaled_dot_product_attention(Qh, Kh, Vh)           # (n_heads, seq_len, d_k)

    concatenated = out_heads.transpose(1, 0, 2).reshape(seq_len, d_model)  # merge heads back
    return concatenated @ Wo                                               # final linear re-projection

rng = np.random.default_rng(0)
seq_len, d_model, n_heads = 6, 16, 4
x = rng.standard_normal((seq_len, d_model)) * 0.1
Wq, Wk, Wv, Wo = (rng.standard_normal((d_model, d_model)) * 0.1 for _ in range(4))

output = multi_head_attention(x, Wq, Wk, Wv, Wo, n_heads)
print("Input shape :", x.shape)
print("Output shape:", output.shape, " (matches input -- ready for the residual connection)")

# Residual connection: identity + learned correction
x_out = x + output
print("\nResidual output x + Sublayer(x), shape:", x_out.shape)
print("If the sublayer output were all zeros, x_out would exactly equal x (identity preserved).")

## 🤖 Why This Matters for AI

Everything above **is** the AI connection — this notebook builds a real (if small) transformer
block. To close the loop, we confirm our from-scratch multi-head attention exactly matches
PyTorch's reference implementation, `torch.nn.functional.scaled_dot_product_attention`, given
identical inputs and weights.

This cross-check matters because `torch.nn.functional.scaled_dot_product_attention` is what every production transformer uses — and when available, it dispatches to **FlashAttention** (Dao et al., 2022), a fused CUDA kernel that computes the exact same mathematics in fewer memory round-trips.

---


In [ ]:
# Cross-checking our from-scratch attention against PyTorch's reference implementation
import numpy as np
import torch
import torch.nn.functional as F

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = np.einsum('...id,...jd->...ij', Q, K) / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return np.einsum('...ij,...jd->...id', weights, V), weights

rng = np.random.default_rng(0)
n_heads, seq_len, d_k = 4, 6, 8
Q = rng.standard_normal((n_heads, seq_len, d_k)).astype(np.float32)
K = rng.standard_normal((n_heads, seq_len, d_k)).astype(np.float32)
V = rng.standard_normal((n_heads, seq_len, d_k)).astype(np.float32)

our_output, _ = scaled_dot_product_attention(Q, K, V)

torch_output = F.scaled_dot_product_attention(
    torch.from_numpy(Q), torch.from_numpy(K), torch.from_numpy(V)
).numpy()

print("Our from-scratch output shape :", our_output.shape)
print("PyTorch reference output shape:", torch_output.shape)
print("Outputs match PyTorch's own implementation:", np.allclose(our_output, torch_output, atol=1e-5))
print("\nEvery concept from this module -- vectors, matrices, transformations, eigenstructure,")
print("SVD, and tensors -- combined into the exact mechanism inside every modern LLM.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For 3 tokens with $Q=\begin{pmatrix}1&0\\0&1\\1&1\end{pmatrix}$,
   $K=\begin{pmatrix}1&0\\0&1\\1&-1\end{pmatrix}$ (so $d_k=2$), compute $QK^\top$ by hand,
   scale by $1/\sqrt{d_k}$, and apply softmax row-wise to get the attention weight matrix.
   Before computing anything, predict the shape of $QK^\top$ from $Q$ and $K$'s shapes alone,
   then confirm every row of your final weight matrix sums to exactly 1.

<details>
<summary>💡 Solution</summary>

Shape prediction: $Q$ is $(3,2)$, $K^\top$ is $(2,3)$, so $QK^\top$ is $(3,3)$ — one score per
(query, key) pair among the 3 tokens. By hand: row 1 of $QK^\top$ is
$(1\cdot1+0\cdot0,\,1\cdot0+0\cdot1,\,1\cdot1+0\cdot(-1))=(1,0,1)$; similarly rows 2 and 3 give
$(0,1,-1)$ and $(1,1,0)$:
$$QK^\top=\begin{pmatrix}1&0&1\\0&1&-1\\1&1&0\end{pmatrix}.$$
Scaling by $1/\sqrt2\approx0.7071$ and applying row-wise softmax gives weights
$\approx\begin{pmatrix}0.401&0.198&0.401\\0.284&0.576&0.140\\0.401&0.401&0.198\end{pmatrix}$,
matching a direct NumPy computation exactly. Each row sums to exactly $1.0$ — softmax's defining
normalization, true by construction for every row regardless of the specific scores.

</details>

### 💭 UNDERSTAND
2. The Theory section writes $Q,K,V$ as `(seq_len, d_k)` matrices, implicitly using the *same*
   dimension $d_k$ for all three. Is that actually required by the attention formula itself?
   Specifically: could $V$'s last dimension differ from $Q$ and $K$'s, while $Q$ and $K$'s last
   dimension must match each other? Reason about which matrix multiplication in
   $\text{softmax}(QK^\top/\sqrt{d_k})V$ forces which shape constraint, then verify by actually
   running attention with $Q,K$ of shape $(5,8)$ and $V$ of shape $(5,3)$.

<details>
<summary>💡 Solution</summary>

$Q$ and $K$'s last dimension *must* match, because $QK^\top$ is a matrix product contracting
exactly that axis — mismatched last dimensions make $QK^\top$ ill-defined, not just numerically
wrong (confirmed: passing $Q$ of shape $(5,8)$ against $K$ of shape $(5,9)$ raises a shape
error immediately, before softmax or anything else even runs). $V$'s last dimension, by
contrast, is completely free to differ: the weights matrix $\text{softmax}(QK^\top/\sqrt{d_k})$
has shape `(seq_len_q, seq_len_k)` regardless of $V$'s shape, and multiplying it by $V$
(`(seq_len_k, d_v)`) only requires $V$'s *first* dimension to match the weights' key-position
axis, not $Q$/$K$'s feature dimension at all. Concretely: $Q,K$ of shape $(5,8)$ with $V$ of
shape $(5,3)$ runs cleanly, producing weights of shape $(5,5)$ (rows still summing to 1) and a
final output of shape $(5,3)$ — matching $V$'s feature dimension, not $Q$/$K$'s. This is exactly
why real transformer implementations are free to use a different head dimension for values than
for queries/keys, if the architecture calls for it.

</details>

### ✏️ DERIVE
3. Show that $G=QQ^\top$ (attention's score matrix in the special case $K=Q$, i.e.
   self-attention comparing tokens against themselves) is always a symmetric,
   positive-semi-definite **Gram matrix**, and that its diagonal entries are exactly the squared
   norms of each token's query vector.

<details>
<summary>💡 Solution outline</summary>

**Symmetry:** $G^\top=(QQ^\top)^\top=(Q^\top)^\top Q^\top=QQ^\top=G$, using
$(AB)^\top=B^\top A^\top$ (Notebook 01.02). **Positive-semi-definiteness:** for any vector
$\mathbf{c}$, $\mathbf{c}^\top G\mathbf{c}=\mathbf{c}^\top QQ^\top\mathbf{c}=
(Q^\top\mathbf{c})^\top(Q^\top\mathbf{c})=\|Q^\top\mathbf{c}\|_2^2\ge0$ — a squared norm can
never be negative, for *any* $Q$, matching the same "$B^\top B$ is always PSD" argument from
`01.06`'s own CHALLENGE exercise, just with $Q^\top$ playing the role of $B$. **Diagonal
entries:** $G_{ii}=(QQ^\top)_{ii}=\sum_d Q_{id}Q_{id}=\sum_d Q_{id}^2=\|\mathbf{q}_i\|_2^2$,
the $i$-th query vector's squared norm exactly. Numeric check on a random $5\times4$ query
matrix: $G$ is symmetric to floating-point precision, its eigenvalues are
$\approx(0,\,0.59,\,5.13,\,8.96,\,10.70)$ — all non-negative (the near-zero one is exactly what's
expected, since a $5\times5$ Gram matrix built from only 4-dimensional vectors must have rank
$\le4$, hence at least one zero eigenvalue) — and $\text{diag}(G)$ matches each row's squared
norm exactly.

</details>

### 🐍 IMPLEMENT
4. Implement a causal version of `scaled_dot_product_attention` by adding $-\infty$ to every
   score at a "future" key position (key index $j>$ query index $i$) before the softmax.

<details>
<summary>✅ How to know you're right</summary>

Every row of your weights matrix must still sum to exactly 1 (masking changes *which* positions
receive weight, not the normalization guarantee). Every entry above the diagonal (query $i$,
key $j>i$) must be *exactly* `0.0`, not just small — verify with `weights[np.triu_indices(seq_len,
k=1)].max() == 0.0`, since $e^{-\infty}=0$ exactly in floating point, not an approximation.
The resulting weight matrix's nonzero region should visibly be lower-triangular (including the
diagonal). As an independent check, your masked output should match
`torch.nn.functional.scaled_dot_product_attention(Q, K, V, is_causal=True)` to floating-point
precision on the same inputs.

</details>

### 🤖 APPLY
5. **Cross-attention** (used in encoder-decoder architectures) has queries come from one
   sequence and keys/values from a *different* sequence of a different length — e.g. a
   4-token decoder sequence attending to a 7-token encoder sequence. Run
   `scaled_dot_product_attention` with `Q` of shape $(4, d_k)$ and `K,V` of shape $(7, d_k)$
   (`V`'s feature dimension can be anything, e.g. 5), and check the shapes at every stage.

<details>
<summary>✅ What you should observe</summary>

The attention weight matrix comes out shape $(4,7)$ — genuinely **non-square**, since it has one
row per decoder (query) position and one column per encoder (key) position, unlike
self-attention's always-square weight matrix. Each of the 4 rows still sums to exactly 1 (every
row is still a softmax over the 7 encoder key positions — the normalization guarantee doesn't
depend on the matrix being square). The final output has shape $(4, 5)$ — one row per decoder
position (matching $Q$'s sequence length), with $V$'s feature dimension. This is the precise
linear-algebra reason cross-attention works at all despite the two sequences having different
lengths: nothing in $\text{softmax}(QK^\top/\sqrt{d_k})V$ ever requires $Q$'s and $K$'s sequence
lengths to match, only their *feature* dimension ($d_k$) — exactly the shape rule from exercise
2, now put to use with two genuinely different sequence lengths instead of one.

</details>

### 🚀 CHALLENGE
6. The attention output is $\text{weights}\times V$, an ordinary matrix product. Investigate
   how the **rank** of the attention output relates to the rank of $V$ alone, independent of
   whatever attention pattern the weights matrix encodes. Fix a 20-token sequence and vary
   $V$'s feature dimension $d_v$ (and hence its maximum possible rank) across several values;
   for each, compute $\text{rank}(V)$, $\text{rank}(\text{weights})$, and
   $\text{rank}(\text{output})$. What structural bound do you find, and why does it hold
   regardless of how "interesting" or varied the attention pattern is?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer finds that $\text{rank}(\text{output})$ always equals $\text{rank}(V)$ in these
tests (for $d_v\in\{2,5,10,20\}$ with a 20-token sequence, output rank tracked $V$'s rank
exactly at each value, and once $d_v$ reached 20, both $V$ and the output saturated at rank 20
and adding more columns to $V$, e.g. $d_v=30$, didn't push the output's rank any higher). The
structural reason: for any matrices $W$ (weights) and $V$, $\text{rank}(WV)\le\min(\text{rank}(W),
\text{rank}(V))$ — a general fact about matrix products, not specific to attention — and since a
softmax-normalized weights matrix built from generic (non-degenerate) scores is essentially
always full rank itself (verified: rank 20 in every test here, regardless of the specific random
$Q,K$ used to build it), the *tighter* constraint in practice is almost always $V$'s own rank,
capping the output's rank at $\min(d_v,\text{seq\_len})$ no matter how sharply peaked, uniform,
or otherwise "interesting" the attention weights are. A strong answer connects this to a genuine
architectural consequence: no amount of clever attention pattern design can inject more
information into the attention output than $V$'s own feature dimension already allows — the
expressive bottleneck for what a single attention operation can *represent* in its output is set
by $d_v$ (and by extension, in multi-head attention, by each head's own value dimension), not by
how sophisticated the query-key matching is.

</details>

---

## 📚 Further Reading
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762)
- [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/)

---

*Next notebook: [Limits and Continuity](../02_Calculus/01_limits_and_continuity.ipynb)*